# Can we predict 30-day readmission?
Design (per checklist.md):
- No calendar column exists, and the same person appears in many rows (1,500 people, ~8 visits each). The test set therefore holds **whole people** (`person_key`), and CV inside training uses grouped folds.
- Dropped: `person_key` (identifier, used only as the group). `visit_number` is kept: it is known at prediction time.
- All other columns are recorded at admission or discharge, i.e. before the 30-day window starts, so they are available at prediction time (prediction made at discharge).
- `postcode_area` is a numeric *code*, so it is treated as categorical.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, f1_score, confusion_matrix
df = pd.read_csv('data.csv')
y = df['readmit_30d']; groups = df['person_key']
X = df.drop(columns=['readmit_30d','person_key'])
print(df.shape, y.mean())

## 1. Hold out a test set of whole people (before any modelling)

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
tr, te = next(gss.split(X, y, groups))
Xtr, ytr, gtr = X.iloc[tr], y.iloc[tr], groups.iloc[tr]
Xte, yte = X.iloc[te], y.iloc[te]
assert set(groups.iloc[tr]).isdisjoint(groups.iloc[te])
print(len(Xtr), len(Xte), ytr.mean(), yte.mean())

## 2. Candidate models (named before any scoring)
1. `baseline`: predicts the training prevalence (trivial).
2. `logreg_C0.1`, `logreg_C1`: logistic regression (scaled numerics, one-hot sex/postcode).
3. `hgb_shallow`, `hgb_deep`: gradient boosting (depth 3 / learning rate 0.05; depth 6 / learning rate 0.1).

Selection metric: mean grouped-CV ROC AUC on training rows. The decision threshold is chosen on out-of-fold predictions by maximising F1; PR-AUC is also reported because the positive class is ~20%.

In [ ]:
num = ['visit_number','birth_year','bmi','length_of_stay_days','n_medications','lab_a','lab_b']
cat = ['sex','postcode_area']
def prep(scale=True):
    return ColumnTransformer([
        ('num', Pipeline([('imp',SimpleImputer(strategy='median'))]+([('sc',StandardScaler())] if scale else [])), num),
        ('cat', Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('oh',OneHotEncoder(handle_unknown='ignore',sparse_output=False))]), cat)])
models = {
 'baseline': Pipeline([('p',prep()),('m',DummyClassifier(strategy='prior'))]),
 'logreg_C0.1': Pipeline([('p',prep()),('m',LogisticRegression(C=0.1,max_iter=2000))]),
 'logreg_C1': Pipeline([('p',prep()),('m',LogisticRegression(C=1,max_iter=2000))]),
 'hgb_shallow': Pipeline([('p',prep(False)),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,random_state=0))]),
 'hgb_deep': Pipeline([('p',prep(False)),('m',HistGradientBoostingClassifier(max_depth=6,learning_rate=0.1,random_state=0))]),
}

## 3. Grouped cross-validation on training rows only (preprocessing is inside each pipeline, so it is fit per fold)

In [ ]:
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)
folds = list(cv.split(Xtr, ytr, gtr))
oof = {}; rows=[]
for n,m in models.items():
    p = np.zeros(len(Xtr)); aucs=[]
    for a,b in folds:
        m.fit(Xtr.iloc[a], ytr.iloc[a]); p[b]=m.predict_proba(Xtr.iloc[b])[:,1]
        aucs.append(roc_auc_score(ytr.iloc[b], p[b]))
    oof[n]=p
    rows.append((n,np.mean(aucs),np.std(aucs),average_precision_score(ytr,p),brier_score_loss(ytr,p)))
cvres = pd.DataFrame(rows,columns=['model','cv_auc_mean','cv_auc_sd','oof_pr_auc','oof_brier']).set_index('model')
cvres

## 4. Choose model and threshold on CV only

In [ ]:
best = cvres['cv_auc_mean'].idxmax()
p = oof[best]
ths = np.linspace(0.05,0.8,76)
f1s = [f1_score(ytr, p>=t) for t in ths]
thr = float(ths[int(np.argmax(f1s))])
print('chosen model:', best, '| threshold:', round(thr,2), '| CV F1:', round(max(f1s),3))
print('prevalence PR-AUC reference =', round(ytr.mean(),3))

**Recorded choice (made from the CV results above, before touching the test set):** the model and threshold printed above; all 9 non-identifier features, no feature selection. If CV AUC is not clearly above 0.5, the conclusion is that readmission is not predictable from these features.

## 5. Refit on all training rows, score the test set once (headline)

In [ ]:
final = models[best].fit(Xtr, ytr)
pt = final.predict_proba(Xte)[:,1]
auc = roc_auc_score(yte, pt)
rng = np.random.default_rng(0)
gte = groups.iloc[te].values; ug = np.unique(gte)
idx = {g:np.where(gte==g)[0] for g in ug}
bs=[]
for _ in range(1000):
    s = np.concatenate([idx[g] for g in rng.choice(ug,len(ug))])
    if yte.iloc[s].nunique()==2: bs.append(roc_auc_score(yte.iloc[s],pt[s]))
print(f'TEST ROC AUC = {auc:.3f}  (95% CI {np.percentile(bs,2.5):.3f}-{np.percentile(bs,97.5):.3f})')
print('TEST PR-AUC =', round(average_precision_score(yte,pt),3), '(prevalence', round(yte.mean(),3),')')
print('TEST Brier =', round(brier_score_loss(yte,pt),4), '| baseline Brier =', round(brier_score_loss(yte,np.full(len(yte),ytr.mean())),4))
print('At CV-chosen threshold: F1 =', round(f1_score(yte,pt>=thr),3)); print(confusion_matrix(yte,pt>=thr))

## Conclusion
The headline is the test ROC AUC above with its person-level bootstrap interval, versus 0.5 for the trivial baseline. Nothing was tuned on the test set. Caveats: there is no calendar date, so generalisation to *future* admissions (rather than *new people*) is untested; the test set has ~300 people, so intervals are wide.